In [ ]:
# spatial_domain_mobilenetv2.py

# Standard MobileNetV2 in Spatial Domain
# Metrics: Accuracy, Cohen's Kappa, Precision, Recall, F1, Specificity, Error Rate

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import os
import warnings
import gc
warnings.filterwarnings('ignore')

# ── Device Setup ────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ================== Step 1: Custom Dataset ==================

class FruitsDataset(Dataset):
    """Custom dataset for Fruits-360 (spatial domain, 3-channel RGB)"""

    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((
                        os.path.join(class_dir, img_name),
                        self.class_to_idx[class_name]
                    ))

        print(f"  Found {len(self.samples)} images across {len(self.classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"  Error loading {img_path}: {e}")
            return torch.zeros(3, 224, 224), label


def load_fruits_dataset(data_root):
    """Load Fruits-360 with standard ImageNet-style transforms"""

    # Fruits-360 images are 100×100; resize to 224×224 for MobileNetV2
    transform_train = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(
            brightness=0.2, contrast=0.2,
            saturation=0.2, hue=0.1
        ),
        transforms.RandomAffine(
            degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    transform_test = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    train_dir = os.path.join(data_root, 'Training')
    test_dir  = os.path.join(data_root, 'Test')

    trainset = FruitsDataset(train_dir, transform=transform_train)
    testset  = FruitsDataset(test_dir,  transform=transform_test)

    return trainset, testset, trainset.classes

# ================== Step 2: Standard MobileNetV2 Model ==================

class SpatialMobileNetV2(nn.Module):
    """
    Standard MobileNetV2 backbone (pretrained) with a custom classifier head.
    Input : 3-channel RGB images (spatial domain — no FFT involved).
    Output: logits over num_classes.
    """

    def __init__(self, num_classes, dropout_rate=0.5):
        super(SpatialMobileNetV2, self).__init__()

        # Load pretrained MobileNetV2
        mobilenet = models.mobilenet_v2(pretrained=True)

        # Keep the original feature extractor unchanged (3-channel input)
        self.features = mobilenet.features          # outputs (B, 1280, 7, 7) for 224×224 input
        last_channels  = mobilenet.last_channel     # 1280

        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Custom classifier head (same depth as baseline for fair comparison)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(last_channels, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )

        self._init_classifier()

    def _init_classifier(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# ================== Step 3: Early Stopping ==================

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, verbose=True):
        self.patience   = patience
        self.min_delta  = min_delta
        self.verbose    = verbose
        self.counter    = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_accuracy, model):
        score = val_accuracy
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f"  EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
            self.counter = 0

# ================== Step 4: Training Function ==================

def train_model(model, train_loader, val_loader,
                epochs=50, lr=0.001, weight_decay=1e-4):
    """
    Train SpatialMobileNetV2.
    - Pretrained backbone uses lr × 0.01 (fine-tune slowly).
    - New classifier head uses lr × 0.5.
    """

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    pretrained_params, new_params = [], []
    for name, param in model.named_parameters():
        if name.startswith('classifier'):
            new_params.append(param)
        else:
            pretrained_params.append(param)

    optimizer = torch.optim.AdamW([
        {'params': pretrained_params, 'lr': lr * 0.01},
        {'params': new_params,        'lr': lr * 0.5}
    ], weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )

    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)

    train_losses, val_losses         = [], []
    train_accuracies, val_accuracies = [], []

    best_val_accuracy = 0.0
    best_model_state  = None

    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

    for epoch in range(epochs):

        # ── Training ────────────────────────────────────────────────────────
        model.train()
        running_loss   = 0.0
        correct_train  = 0
        total_train    = 0

        train_pbar = tqdm(
            train_loader,
            desc=f"Epoch {epoch+1}/{epochs} [Train]",
            leave=False
        )

        for i, (images, labels) in enumerate(train_pbar):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss    = criterion(outputs, labels)

                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"  Warning: NaN/Inf loss at batch {i}, skipping.")
                    continue

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)

                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"  Warning: NaN/Inf loss at batch {i}, skipping.")
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            running_loss  += loss.item()
            _, predicted   = torch.max(outputs.data, 1)
            total_train   += labels.size(0)
            correct_train += (predicted == labels).sum().item()

            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc' : f'{100 * correct_train / total_train:.2f}%'
            })

            if i % 50 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()

        scheduler.step()

        avg_train_loss  = running_loss  / max(len(train_loader), 1)
        train_accuracy  = 100 * correct_train / max(total_train, 1)
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)

        # ── Validation ──────────────────────────────────────────────────────
        model.eval()
        running_val_loss = 0.0
        correct  = 0
        total    = 0

        with torch.no_grad():
            val_pbar = tqdm(
                val_loader,
                desc=f"Epoch {epoch+1}/{epochs} [Val]",
                leave=False
            )
            for images, labels in val_pbar:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                if scaler is not None:
                    with torch.amp.autocast('cuda'):
                        outputs = model(images)
                        loss    = criterion(outputs, labels)
                else:
                    outputs = model(images)
                    loss    = criterion(outputs, labels)

                running_val_loss += loss.item()
                _, predicted      = torch.max(outputs.data, 1)
                total   += labels.size(0)
                correct += (predicted == labels).sum().item()

                val_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc' : f'{100 * correct / total:.2f}%'
                })

        avg_val_loss   = running_val_loss / max(len(val_loader), 1)
        val_accuracy   = 100 * correct / max(total, 1)
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state  = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }

        print(f"\nEpoch [{epoch+1}/{epochs}]")
        print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.2f}%")
        print(f"  Val   Loss: {avg_val_loss:.4f} | Val   Acc: {val_accuracy:.2f}%")
        print(f"  LR (backbone): {optimizer.param_groups[0]['lr']:.2e}  "
              f"LR (head): {optimizer.param_groups[1]['lr']:.2e}")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("  Early stopping triggered!")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nRestored best model — Val Acc: {best_val_accuracy:.2f}%")

    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 5: Comprehensive Metrics Computation ==================

def compute_all_metrics(all_labels, all_predictions, num_classes):
    """
    Compute and return a dictionary of all required metrics.

    Metrics computed
    ----------------
    1. Overall Accuracy
    2. Cohen's Kappa Score
    3. Overall Precision  (macro)
    4. Overall Recall     (macro)
    5. Overall F1 Score   (macro)
    6. Overall Specificity (macro-average via confusion matrix)
    7. Overall Error Rate
    """

    all_labels      = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # 1. Accuracy
    accuracy = accuracy_score(all_labels, all_predictions) * 100

    # 2. Cohen's Kappa
    kappa = cohen_kappa_score(all_labels, all_predictions)

    # 3. Precision (macro)
    precision = precision_score(
        all_labels, all_predictions,
        average='macro', zero_division=0
    ) * 100

    # 4. Recall (macro)
    recall = recall_score(
        all_labels, all_predictions,
        average='macro', zero_division=0
    ) * 100

    # 5. F1 Score (macro)
    f1 = f1_score(
        all_labels, all_predictions,
        average='macro', zero_division=0
    ) * 100

    # 6. Specificity (macro-average)
    # For each class c:  specificity_c = TN_c / (TN_c + FP_c)
    # where TN_c = all correct non-c predictions,
    #       FP_c = non-c samples predicted as c.
    cm = confusion_matrix(all_labels, all_predictions, labels=list(range(num_classes)))

    specificities = []
    for c in range(num_classes):
        TP = cm[c, c]
        FP = cm[:, c].sum() - TP          # column sum minus diagonal
        FN = cm[c, :].sum() - TP          # row sum minus diagonal
        TN = cm.sum() - TP - FP - FN

        denom = TN + FP
        spec_c = TN / denom if denom > 0 else 0.0
        specificities.append(spec_c)

    specificity = np.mean(specificities) * 100   # macro-average

    # 7. Error Rate
    error_rate = 100 - accuracy

    metrics = {
        'Overall Accuracy (%)':    accuracy,
        "Cohen's Kappa":           kappa,
        'Overall Precision (%)':   precision,
        'Overall Recall (%)':      recall,
        'Overall F1 Score (%)':    f1,
        'Overall Specificity (%)': specificity,
        'Overall Error Rate (%)':  error_rate,
    }

    return metrics, cm


def print_metrics(metrics):
    """Pretty-print the metrics dictionary."""
    print("\n" + "=" * 55)
    print("        COMPREHENSIVE EVALUATION METRICS")
    print("=" * 55)
    for name, value in metrics.items():
        if name == "Cohen's Kappa":
            print(f"  {name:<30s}: {value:.4f}")
        else:
            print(f"  {name:<30s}: {value:.2f}")
    print("=" * 55)


def plot_confusion_matrix_summary(cm, num_classes, max_classes_shown=20):
    """
    Plot a truncated confusion matrix (first max_classes_shown classes)
    to keep the figure readable for large datasets.
    """
    n = min(num_classes, max_classes_shown)
    cm_sub = cm[:n, :n]

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_sub, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_title(
        f'Confusion Matrix (first {n} of {num_classes} classes)\n'
        'Spatial Domain MobileNetV2',
        fontsize=14, fontweight='bold'
    )
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label',      fontsize=12)

    thresh = cm_sub.max() / 2.0
    for i in range(n):
        for j in range(n):
            ax.text(j, i, str(cm_sub[i, j]),
                    ha='center', va='center', fontsize=6,
                    color='white' if cm_sub[i, j] > thresh else 'black')

    plt.tight_layout()
    plt.show()


def plot_training_curves(train_losses, val_losses,
                         train_accuracies, val_accuracies):
    """Plot loss and accuracy curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(train_losses) + 1)

    ax1.plot(epochs, train_losses, 'b-', label='Train Loss',      linewidth=2)
    ax1.plot(epochs, val_losses,   'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch');  ax1.set_ylabel('Loss')
    ax1.set_title('Loss Curves', fontsize=14, fontweight='bold')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, train_accuracies, 'b-', label='Train Acc',      linewidth=2)
    ax2.plot(epochs, val_accuracies,   'r-', label='Validation Acc', linewidth=2)
    ax2.set_xlabel('Epoch');  ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Accuracy Curves', fontsize=14, fontweight='bold')
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_metrics_bar(metrics):
    """Bar chart of all computed metrics."""
    # Exclude Cohen's Kappa from the percentage bar (different scale)
    pct_metrics = {k: v for k, v in metrics.items() if k != "Cohen's Kappa"}
    kappa_val   = metrics["Cohen's Kappa"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Spatial Domain MobileNetV2 — Evaluation Metrics',
                 fontsize=15, fontweight='bold')

    # Left: percentage metrics
    names  = [k.replace(' (%)', '') for k in pct_metrics.keys()]
    values = list(pct_metrics.values())
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#00BCD4']

    bars = axes[0].bar(names, values, color=colors[:len(names)], edgecolor='black', linewidth=0.8)
    axes[0].set_ylim(0, 110)
    axes[0].set_ylabel('Score (%)', fontsize=12)
    axes[0].set_title('Performance Metrics (%)', fontsize=13, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=25)
    axes[0].grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, values):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val:.2f}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold'
        )

    # Right: Cohen's Kappa (0-1 scale)
    axes[1].bar(["Cohen's Kappa"], [kappa_val],
                color='#607D8B', edgecolor='black', linewidth=0.8)
    axes[1].set_ylim(0, 1.05)
    axes[1].set_ylabel('Kappa Score', fontsize=12)
    axes[1].set_title("Cohen's Kappa", fontsize=13, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)
    axes[1].text(0, kappa_val + 0.01, f'{kappa_val:.4f}',
                 ha='center', va='bottom', fontsize=12, fontweight='bold')

    # Kappa interpretation band
    for lo, hi, label, color in [
        (0.0, 0.2, 'Slight',      '#FFCDD2'),
        (0.2, 0.4, 'Fair',        '#FFE0B2'),
        (0.4, 0.6, 'Moderate',    '#FFF9C4'),
        (0.6, 0.8, 'Substantial', '#C8E6C9'),
        (0.8, 1.0, 'Almost\nPerfect', '#BBDEFB'),
    ]:
        axes[1].axhspan(lo, hi, alpha=0.15, color=color, label=label)
    axes[1].legend(loc='lower right', fontsize=8, title='Interpretation')

    plt.tight_layout()
    plt.show()

# ================== Step 6: Main Execution Pipeline ==================

def main():
    print("=" * 70)
    print("   Spatial Domain MobileNetV2 — Fruits-360 Classification")
    print("   Metrics: Accuracy | Kappa | Precision | Recall |")
    print("            F1 | Specificity | Error Rate")
    print("=" * 70)

    data_root = r'C:\Users\CSE_SDPL\Downloads\fruits-360_100x100\fruits-360'

    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found:\n  {data_root}")
        print("Please update 'data_root' to point to your Fruits-360 folder.")
        return

    # ── Step 1: Load Dataset ─────────────────────────────────────────────────
    print("\n[Step 1] Loading Fruits-360 dataset...")
    trainset, testset, classes = load_fruits_dataset(data_root)
    num_classes = len(classes)
    print(f"  Number of classes : {num_classes}")

    # ── Step 2: Train / Val Split ────────────────────────────────────────────
    print("\n[Step 2] Splitting training set (85% train / 15% val)...")
    train_size = int(0.85 * len(trainset))
    val_size   = len(trainset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        trainset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    print(f"  Train : {len(train_subset)}")
    print(f"  Val   : {len(val_subset)}")
    print(f"  Test  : {len(testset)}")

    # ── Step 3: DataLoaders ──────────────────────────────────────────────────
    print("\n[Step 3] Building DataLoaders...")
    batch_size  = 64          # slightly lower than freq version (224×224 images)
    num_workers = 4 if os.name != 'nt' else 0
    print(f"  Batch size  : {batch_size}")
    print(f"  Num workers : {num_workers}")

    train_loader = DataLoader(
        train_subset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True,
        prefetch_factor=2 if num_workers > 0 else None
    )
    val_loader = DataLoader(
        val_subset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        prefetch_factor=2 if num_workers > 0 else None
    )
    test_loader = DataLoader(
        testset, batch_size=64, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        prefetch_factor=2 if num_workers > 0 else None
    )

    # ── Step 4: Build Model ──────────────────────────────────────────────────
    print("\n[Step 4] Building Spatial MobileNetV2 model...")
    model = SpatialMobileNetV2(num_classes=num_classes, dropout_rate=0.5).to(device)
    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total parameters    : {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")

    if torch.cuda.is_available():
        print(f"  GPU memory allocated: "
              f"{torch.cuda.memory_allocated(0)/1e9:.2f} GB")

    # ── Step 5: Train ────────────────────────────────────────────────────────
    print("\n[Step 5] Training model...")
    train_losses, val_losses, train_accuracies, val_accuracies = train_model(
        model, train_loader, val_loader,
        epochs=50, lr=0.001, weight_decay=5e-4
    )

    print("\n[Step 5.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses,
                         train_accuracies, val_accuracies)

    # ── Step 6: Evaluate on Test Set ─────────────────────────────────────────
    print("\n[Step 6] Evaluating on the test set...")
    model.eval()

    all_predictions = []
    all_labels      = []
    all_confidences = []

    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc="Testing")
        for images, labels in test_pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs      = model(images)
            probs        = F.softmax(outputs, dim=1)
            conf, pred   = torch.max(probs, dim=1)

            all_predictions.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_confidences.extend(conf.cpu().numpy())

            running_correct = (np.array(all_predictions) ==
                               np.array(all_labels)).sum()
            test_pbar.set_postfix({
                'acc': f'{100 * running_correct / len(all_labels):.2f}%'
            })

    # ── Step 7: Compute & Display Metrics ────────────────────────────────────
    print("\n[Step 7] Computing comprehensive evaluation metrics...")
    metrics, cm = compute_all_metrics(all_labels, all_predictions, num_classes)
    print_metrics(metrics)

    # Average confidence
    avg_conf = np.mean(all_confidences) * 100
    print(f"\n  Average Prediction Confidence: {avg_conf:.2f}%")

    # Per-class report (printed to console, not stored)
    print("\n[Step 7.1] Per-class Classification Report (first 20 classes):")
    target_names_short = [classes[i] for i in range(min(num_classes, 20))]
    idx_mask = np.isin(all_labels, list(range(min(num_classes, 20))))
    if idx_mask.sum() > 0:
        print(classification_report(
            np.array(all_labels)[idx_mask],
            np.array(all_predictions)[idx_mask],
            target_names=target_names_short,
            zero_division=0
        ))

    # ── Step 8: Visualizations ────────────────────────────────────────────────
    print("\n[Step 8] Generating visualizations...")
    plot_metrics_bar(metrics)
    plot_confusion_matrix_summary(cm, num_classes, max_classes_shown=20)

    # ── Step 9: Save Model ────────────────────────────────────────────────────
    print("\n[Step 9] Saving trained model...")
    save_path = 'fruits_spatial_mobilenetv2.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'metrics'         : metrics,
        'classes'         : classes,
        'num_classes'     : num_classes,
        'train_losses'    : train_losses,
        'val_losses'      : val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies'  : val_accuracies,
    }, save_path)
    print(f"  Model saved → '{save_path}'")

    # ── Final Summary ─────────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("  FINAL RESULTS — Spatial Domain MobileNetV2")
    print("=" * 70)
    print_metrics(metrics)
    print(f"\n  Model : Spatial MobileNetV2 (pretrained ImageNet)")
    print(f"  Domain: RGB Spatial (no FFT)")
    print(f"  Data  : Fruits-360 | {num_classes} classes")
    print("=" * 70)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"\n  Final GPU memory allocated: "
              f"{torch.cuda.memory_allocated(0)/1e9:.2f} GB")


if __name__ == "__main__":
    main()